In [1]:
import json
import os
import pandas as pd
import glob
from transformers import AutoTokenizer
import re
import json
os.chdir('..')

/raid/home/m13521157/absa-eap-ig/enveap/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '7'

In [3]:
from typing import List, Dict
def parse_absa_string(text: str) -> List[Dict[str, str]]:
    """
    Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
    Each dictionary contains the tag as the key and the corresponding value.
    For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
    [{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
    {'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

    Args:
        text (str): ABSA string output to be parsed.

    Returns:
        List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

    """
    pattern = r"\[(\w+)\]\s*([^[]+)"
    matches = re.findall(pattern, text)

    result = []
    current_dict = {}

    for tag, content in matches:
        if tag == "SSEP":  # Sentence separator -> Start a new dictionary
            result.append(current_dict)
            current_dict = {}
        else:
            current_dict[tag] = content.strip()

    if current_dict:  # Append the last sentence if it exists
        result.append(current_dict)

    return result

## Construct from previous Gas and Legoabsa

### Gas

In [3]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

In [4]:
tokenizer.all_special_tokens

['<|endoftext|>',
 '<|im_start|>',
 '<|im_end|>',
 '<|object_ref_start|>',
 '<|object_ref_end|>',
 '<|box_start|>',
 '<|box_end|>',
 '<|quad_start|>',
 '<|quad_end|>',
 '<|vision_start|>',
 '<|vision_end|>',
 '<|vision_pad|>',
 '<|image_pad|>',
 '<|video_pad|>']

### Lego Absa

In [6]:
import glob

# List all directories matching the pattern
for lang in ['eng', 'indo', 'sunda']:
	directories = [d for d in glob.glob(f'hotel_dataset/{lang}/old/clean_train2500_lego_absa*') if os.path.isdir(d)]
	print(directories)

['hotel_dataset/eng/old/clean_train2500_lego_absa_multitask_task_transfer', 'hotel_dataset/eng/old/clean_train2500_lego_absa_task_transfer', 'hotel_dataset/eng/old/clean_train2500_lego_absa_multitask_indolegoabsa', 'hotel_dataset/eng/old/clean_train2500_lego_absa_multitask']
['hotel_dataset/indo/old/clean_train2500_lego_absa_multitask_task_transfer', 'hotel_dataset/indo/old/clean_train2500_lego_absa_task_transfer', 'hotel_dataset/indo/old/clean_train2500_lego_absa_multitask_indolegoabsa', 'hotel_dataset/indo/old/clean_train2500_lego_absa_multitask']
['hotel_dataset/sunda/old/clean_train2500_lego_absa_multitask_task_transfer', 'hotel_dataset/sunda/old/clean_train2500_lego_absa_task_transfer', 'hotel_dataset/sunda/old/clean_train2500_lego_absa_multitask_indolegoabsa', 'hotel_dataset/sunda/old/clean_train2500_lego_absa_multitask']


In [5]:
os.path.basename(directories[0])

NameError: name 'directories' is not defined

In [25]:
# aspect: '<|box_start|>'
# opinion: '<|quad_start|>'
# sentiment: '<|vision_start|>'

In [26]:
import re
from tqdm import tqdm

for lang in ['eng', 'indo', 'sunda']:
	directories = [d for d in glob.glob(f'hotel_dataset/{lang}/old/clean_train2500_lego_absa*') if os.path.isdir(d)]
	for directory in directories:
		files = os.listdir(directory)
		files = [os.path.join(directory, f) for f in files]
		special_tokens = {
			'a': '<|box_start|>',
			'o': '<|quad_start|>',
			's': '<|vision_start|>'
		}
		for file in files:
			if file.endswith('.json'):
				with open(file, 'r') as f:
					data = json.load(f)
				new_data = []
				for instance in tqdm(data):
					sentence = instance['input']
					targets = instance['target']
					sentence = sentence.replace('aspect : <|endoftext|>', 'aspect: <|box_start|>')
					sentence = sentence.replace('opinion : <|endoftext|>', 'opinion: <|quad_start|>')
					sentence = sentence.replace('sentiment : <|endoftext|>', 'sentiment: <|vision_start|>')
					sentence = sentence.replace('<|endoftext|>', '').strip()
					targets = targets.split(';')
					new_targets = []
					for index, target in enumerate(targets):
						target = target.strip()
						matches = re.findall(r'<\|endoftext\|>', target)
						assert len(matches) == len(instance['element_order']) + 1
						target_temp = list(target)
						for element in instance['element_order']:
							target = ''.join(target_temp)
							match = re.search(r'<\|endoftext\|>', target)
							if match:
								target_temp[match.start():match.end()] = special_tokens[element]
							else:
								raise ValueError(f"No match found for <|endoftext|> in target: {target}")
						# Remove any remaining <|endoftext|> tokens
						new_target = ''.join(target_temp)
						new_target = re.sub(r'<\|endoftext\|>', '', new_target)
						new_target = new_target.strip()
						new_targets.append(new_target)
					new_targets = ';'.join(new_targets)
					new_data.append({
						"sentence_id": instance['sentence_id'],
						"instance_id": instance['instance_id'],
						"task_elements": instance['task_elements'],
						"input": sentence,
						"target": new_targets,
						"element_order": instance['element_order']
					})
				# Write the modified data to a new JSON file
				new_path = os.path.join('hotel_dataset', lang, os.path.basename(directory), os.path.basename(file))
				os.makedirs(os.path.dirname(new_path), exist_ok=True)
				with open(new_path, 'w') as f:
					json.dump(new_data, f, indent=4, ensure_ascii=False)
				print(f"Processed {file} and saved to {new_path}")

  0%|          | 0/7500 [00:00<?, ?it/s]

100%|██████████| 7500/7500 [00:00<00:00, 41062.39it/s]


Processed hotel_dataset/eng/old/clean_train2500_lego_absa_multitask_task_transfer/hotel_aste_train_augmented_noreasoning.json and saved to hotel_dataset/eng/clean_train2500_lego_absa_multitask_task_transfer/hotel_aste_train_augmented_noreasoning.json


100%|██████████| 1000/1000 [00:00<00:00, 39238.71it/s]


Processed hotel_dataset/eng/old/clean_train2500_lego_absa_multitask_task_transfer/hotel_aste_dev_augmented.json and saved to hotel_dataset/eng/clean_train2500_lego_absa_multitask_task_transfer/hotel_aste_dev_augmented.json


...

Processed hotel_dataset/sunda/old/clean_train2500_lego_absa_multitask/hotel_aste_train_augmented_noreasoning.json and saved to hotel_dataset/sunda/clean_train2500_lego_absa_multitask/hotel_aste_train_augmented_noreasoning.json


100%|██████████| 1000/1000 [00:00<00:00, 40025.04it/s]


Processed hotel_dataset/sunda/old/clean_train2500_lego_absa_multitask/hotel_aste_dev_augmented.json and saved to hotel_dataset/sunda/clean_train2500_lego_absa_multitask/hotel_aste_dev_augmented.json


100%|██████████| 1000/1000 [00:00<00:00, 37468.55it/s]

Processed hotel_dataset/sunda/old/clean_train2500_lego_absa_multitask/hotel_aste_test_augmented.json and saved to hotel_dataset/sunda/clean_train2500_lego_absa_multitask/hotel_aste_test_augmented.json


In [13]:
new_data

[{'sentence_id': 0,
  'instance_id': 0,
  'task_elements': 'aos',
  'input': 'the service is very friendly .| opinion: <|quad_start|> , aspect: <|box_start|> , sentiment: <|vision_start|>',
  'target': '<|quad_start|> very friendly <|box_start|> service <|vision_start|> positive',
  'element_order': 'oas'},
 {'sentence_id': 1,
  'instance_id': 1,
  'task_elements': 'aos',
  'input': "it's a shame the wifi isn't good ; i have to go outside the room .| opinion: <|quad_start|> , aspect: <|box_start|> , sentiment: <|vision_start|>",
  'target': "<|quad_start|> isn't good <|box_start|> wifi <|vision_start|> negative",
  'element_order': 'oas'},
 {'sentence_id': 2,
  'instance_id': 2,
  'task_elements': 'aos',
  'input': 'the description said twin bed , but the room had a different bed setup .| opinion: <|quad_start|> , aspect: <|box_start|> , sentiment: <|vision_start|>',
  'target': '<|quad_start|> different <|box_start|> room <|vision_start|> negative',
  'element_order': 'oas'},
 {'sente

In [24]:
test = list('<|endoftext|> not')
test[0:13] = ['a', 'b', 'c']
test

['a', 'b', 'c', ' ', 'n', 'o', 't']

## Construct from MvP format

### GAS

In [12]:
lang = 'indo'
dataset_folder = 'corrected_splitopinion_typocorrected'
dataset_per_split = {}
splits = ['train', 'test']

In [13]:
# Read the modified files and combine them into train, dev, test splits
for split in splits:
	dataset_per_split[split] = []
	files = glob.glob(f'hotel_dataset/{lang}/{dataset_folder}/*{split}*.json')
	print("Reading files for", split, ":", files)
	for file in files:
		with open(file, 'r') as f:
			data = json.load(f)
		dataset_per_split[split].extend(data)

Reading files for train : ['hotel_dataset/indo/corrected_splitopinion_typocorrected/hotel_aste_train_augmented_noreasoning.json']
Reading files for test : ['hotel_dataset/indo/corrected_splitopinion_typocorrected/hotel_aste_test_augmented.json']


In [14]:
unique_data_per_split = {}
for split in splits:
	unique_data_per_split[split] = []
	for i in range(0, len(dataset_per_split[split]), 5):
		instance = dataset_per_split[split][i]
		unique_data_per_split[split].append(instance)

In [15]:
def convert_to_gas_format(data_list):
	"""
	Converts a list of aspect-based sentiment dictionaries to the 
	extraction-style string format used in the GAS paper.

	Args:
		data_list: A list of dictionaries, where each dictionary must contain 
				the keys 'A' (Aspect), 'O' (Opinion), and 'S' (Sentiment).

	Returns:
		A single string formatted as '(A, O, S); (A, O, S); ...'
	"""
	# A list comprehension to create a formatted string for each dictionary
	# The order of elements in the tuple is specified as A, O, S
	triplets = [f"({item['A']}, {item['O']}, {item['S']})" for item in data_list]
	
	# Join the list of strings together with a semicolon and space
	return "; ".join(triplets)

In [16]:
gas_data_per_split = {}
for split in splits:
	gas_data_per_split[split] = []
	instance_id_counter = unique_data_per_split[split][0]['sentence_id']
	for instance in unique_data_per_split[split]:
		absa_string = instance['target']
		parsed_absa = parse_absa_string(absa_string)
		try:
			for d in parsed_absa:
				assert all(key in d.keys() for key in ['A', 'O', 'S'])
		except:
			print("Error in instance:", instance)
			raise ValueError("Parsed ABSA contains invalid keys.")
		gas_format = convert_to_gas_format(parsed_absa)
		gas_data_per_split[split].append({
			"sentence_id": instance['sentence_id'],
			"instance_id": instance_id_counter,
			'task_elements': instance['task_elements'],
			"input": f"{instance['input'].replace('[A] [O] [S]', '').strip()}", # Add arrow for input target delimiter (remove if not needed)
			"target": gas_format.strip(),
			"element_order": instance['element_order']
		})
		instance_id_counter += 1

In [17]:
len(gas_data_per_split['train']), len(gas_data_per_split['test'])

(2482, 1000)

In [18]:
for split in splits:
    gas_dataset_path = f'hotel_dataset/{lang}/corrected_splitopinion_typocorrected_gas'
    os.makedirs(gas_dataset_path, exist_ok=True)
    with open(f"hotel_dataset/{lang}/corrected_splitopinion_typocorrected_gas/hotel_aste_{split}_augmented{'_noreasoning' if split == 'train' else ''}.json", 'w') as f:
        json.dump(gas_data_per_split[split], f, ensure_ascii=False, indent=4)

### Lego-ABSA

In [139]:
lang = 'indo'
dataset_folder = 'corrected_splitopinion_typocorrected'
dataset_per_split = {}
splits = ['train', 'test']

In [140]:
# Read the modified files and combine them into train, dev, test splits
for split in splits:
	dataset_per_split[split] = []
	files = glob.glob(f'hotel_dataset/{lang}/{dataset_folder}/*{split}*.json')
	print("Reading files for", split, ":", files)
	for file in files:
		with open(file, 'r') as f:
			data = json.load(f)
		dataset_per_split[split].extend(data)

Reading files for train : ['hotel_dataset/indo/corrected_splitopinion_typocorrected/hotel_aste_train_augmented_noreasoning.json']
Reading files for test : ['hotel_dataset/indo/corrected_splitopinion_typocorrected/hotel_aste_test_augmented.json']


In [141]:
unique_data_per_split = {}
for split in splits:
	unique_data_per_split[split] = []
	for i in range(0, len(dataset_per_split[split]), 5):
		instance = dataset_per_split[split][i]
		unique_data_per_split[split].append(instance)

In [142]:
element_orders = ['oa', 'as']

In [143]:
special_tokens = {
	'a': '<|box_start|>',
	'o': '<|quad_start|>',
	's': '<|vision_start|>'
}

initial_definitions = {
    'a': 'aspect',
    'o': 'opinion',
    's': 'sentiment'
}

In [144]:
def convert_output_to_legoabsa_format(data_list, order='aos'):
	"""
	Converts a list of aspect-based sentiment dictionaries to the 
	extraction-style string format used in the Lego-ABSA paper.

	Args:
		data_list: A list of strings, where each dictionary must contain 
				the keys 'A' (Aspect), 'O' (Opinion), and 'S' (Sentiment).
		order: A string specifying the order of elements (e.g., 'aos', 'ao', 'as', 'a', 'o')

	Returns:
		A single string formatted with special tokens and elements based on the order
	"""
	global special_tokens
	# Convert order string to list of characters
	tuple_order = list(order.lower())
	
	# Build triplets based on the order length
	triplets = []
	for item in data_list:
		parts = []
		for element in tuple_order:
			if element.upper() in item:
				parts.append(f"{special_tokens[element]} {item[element.upper()].strip()}")
		triplets.append(" ".join(parts))
	
	# Join the list of strings together with a semicolon and space
	return ";".join(triplets)

def convert_input_to_legoabsa_format(input_str, order='aos'):
	global special_tokens
	global initial_definitions
	# Convert order string to list of characters
	tuple_order = list(order.lower())
	
	input_str = input_str.replace('[A] [O] [S]', '').strip()
	
	# Make replacements based on the order like this '{input_str}|aspect: <|box_start|> , opinion: <|quad_start|> , sentiment: <|vision_start|>'
	definitions = ' , '.join([f"{initial_definitions[element]}: {special_tokens[element]}" for element in tuple_order])
	return f"{input_str}| {definitions}"

In [145]:
convert_input_to_legoabsa_format("The room was clean but the service was terrible. The location is great though.", order='sa')

'The room was clean but the service was terrible. The location is great though.| sentiment: <|vision_start|> , aspect: <|box_start|>'

In [146]:
convert_output_to_legoabsa_format([{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
 {'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}], order='aos')

'<|box_start|> harga <|quad_start|> terjangkau <|vision_start|> positive;<|box_start|> fasilitas <|quad_start|> nyaman <|vision_start|> positive'

In [147]:
lego_absa_data_per_split = {}
lego_absa_data_per_split['train'] = []
instance_id_counter = unique_data_per_split['train'][0]['sentence_id']
for element_order in element_orders:
	for instance in unique_data_per_split['train']:
		absa_string = instance['target']
		parsed_absa = parse_absa_string(absa_string)
		try:
			for d in parsed_absa:
				assert all(key in d.keys() for key in ['A', 'O', 'S'])
		except:
			print("Error in instance:", instance)
			raise ValueError("Parsed ABSA contains invalid keys.")
		lego_absa_format = convert_output_to_legoabsa_format(parsed_absa, order=element_order)
		input_legoabsa_format = convert_input_to_legoabsa_format(instance['input'], order=element_order)
		lego_absa_data_per_split['train'].append({
			"sentence_id": instance['sentence_id'],
			"instance_id": instance_id_counter,
			'task_elements': instance['task_elements'],
			"input": input_legoabsa_format.strip(),
			"target": lego_absa_format.strip(),
			"element_order": element_order
		})
		instance_id_counter += 1

In [148]:
lego_absa_data_per_split['train'].__len__()

4964

In [149]:
lego_absa_data_per_split['test'] = []
instance_id_counter = len(lego_absa_data_per_split['train']) + 1000 # Start test IDs after train IDs + dev IDs (dev have 1000 instances)
for instance in unique_data_per_split['test']:
	absa_string = instance['target']
	parsed_absa = parse_absa_string(absa_string)
	try:
		for d in parsed_absa:
			assert all(key in d.keys() for key in ['A', 'O', 'S'])
	except:
		print("Error in instance:", instance)
		raise ValueError("Parsed ABSA contains invalid keys.")
	lego_absa_format = convert_output_to_legoabsa_format(parsed_absa, order='aos')
	input_legoabsa_format = convert_input_to_legoabsa_format(instance['input'], order='aos')
	lego_absa_data_per_split['test'].append({
		"sentence_id": instance['sentence_id'],
		"instance_id": instance_id_counter,
		'task_elements': instance['task_elements'],
		"input": input_legoabsa_format.strip(),
		"target": lego_absa_format.strip(),
		"element_order": 'aos'
	})
	instance_id_counter += 1

In [150]:
lego_absa_data_per_split['train'][:2]

[{'sentence_id': 0,
  'instance_id': 0,
  'task_elements': 'aos',
  'input': 'kamar saya ada kendala di ac tidak berfungsi optimal . dan juga wifi koneksi kurang stabil .| opinion: <|quad_start|> , aspect: <|box_start|>',
  'target': '<|quad_start|> tidak berfungsi optimal <|box_start|> ac;<|quad_start|> kurang stabil <|box_start|> wifi koneksi',
  'element_order': 'oa'},
 {'sentence_id': 1,
  'instance_id': 1,
  'task_elements': 'aos',
  'input': 'tempatnya bagus . kolam renangnya bersih .| opinion: <|quad_start|> , aspect: <|box_start|>',
  'target': '<|quad_start|> bagus <|box_start|> tempatnya;<|quad_start|> bersih <|box_start|> kolam renangnya',
  'element_order': 'oa'}]

In [151]:
lego_absa_data_per_split['test'][:2]

[{'sentence_id': 3500,
  'instance_id': 5964,
  'task_elements': 'aos',
  'input': 'pelayanan nya sangat ramah .| aspect: <|box_start|> , opinion: <|quad_start|> , sentiment: <|vision_start|>',
  'target': '<|box_start|> pelayanan nya <|quad_start|> sangat ramah <|vision_start|> positive',
  'element_order': 'aos'},
 {'sentence_id': 3501,
  'instance_id': 5965,
  'task_elements': 'aos',
  'input': 'sayang wifi tidak bagus harus keluar kamar .| aspect: <|box_start|> , opinion: <|quad_start|> , sentiment: <|vision_start|>',
  'target': '<|box_start|> wifi <|quad_start|> tidak bagus harus keluar kamar <|vision_start|> negative',
  'element_order': 'aos'}]

In [152]:
for split in splits:
    legoabsa_dataset_path = f'hotel_dataset/{lang}/corrected_splitopinion_typocorrected_tasktransfer_legoabsa'
    os.makedirs(legoabsa_dataset_path, exist_ok=True)
    with open(f"hotel_dataset/{lang}/corrected_splitopinion_typocorrected_tasktransfer_legoabsa/hotel_aste_{split}_augmented{'_noreasoning' if split == 'train' else ''}.json", 'w') as f:
        json.dump(lego_absa_data_per_split[split], f, ensure_ascii=False, indent=4)